# Proyek Machine Learning Pipeline: Mushroom Classification

Proyek ini membangun pipeline machine learning end-to-end menggunakan 
**TensorFlow Extended (TFX)** untuk mengklasifikasi jamur sebagai 
beracun (*poisonous*) atau dapat dimakan (*edible*).

**Dataset:** Mushroom Classification - UCI ML Repository (8.124 sampel, 22 fitur kategorikal)  
**Persoalan:** Identifikasi jamur beracun secara manual sulit dan berisiko fatal.  
**Solusi:** Binary classifier neural network dengan pipeline TFX otomatis.  
**Target:** BinaryAccuracy ≥ 0.95 dan AUC ≥ 0.95  
**Label:** `class` → `p` (poisonous=1) atau `e` (edible=0)

## Import Library

In [1]:
import os
import tfx
import tensorflow as tf
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    Evaluator,
    ExampleValidator,
    Pusher,
    SchemaGen,
    StatisticsGen,
    Trainer,
    Transform,
    Tuner,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)
from tfx.proto import example_gen_pb2, pusher_pb2, trainer_pb2, tuner_pb2
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

print('TensorFlow version:', tf.__version__)
print('TFX version:', tfx.__version__)

TensorFlow version: 2.10.1
TFX version: 1.11.0


## Konfigurasi Pipeline

Menambahkan `TUNER_MODULE_FILE` untuk modul Tuner baru.

In [2]:
PIPELINE_NAME         = "yogidharma-pipeline"
PIPELINE_ROOT         = PIPELINE_NAME
DATA_ROOT             = "data"
TRANSFORM_MODULE_FILE = "modules/mushroom_transform.py"
TRAINER_MODULE_FILE   = "modules/mushroom_trainer.py"
TUNER_MODULE_FILE     = "modules/mushroom_tuner.py"   # ← BARU
SERVING_MODEL_DIR     = "serving_model_dir/mushroom_model"

interactive_context = InteractiveContext(
    pipeline_root=PIPELINE_ROOT
)

print('Pipeline root :', PIPELINE_ROOT)
print('Serving model :', SERVING_MODEL_DIR)

Pipeline root : yogidharma-pipeline
Serving model : serving_model_dir/mushroom_model


## 1. ExampleGen — Ingesti Data

Komponen pertama pipeline yang membaca data mentah CSV dan mengkonversinya 
ke format TFRecord. Data dibagi secara deterministik:
- **Train (80%)**: 8 hash buckets untuk pelatihan
- **Eval (20%)**: 2 hash buckets untuk evaluasi

In [3]:
output = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval",  hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output)
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 22
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

## 2. StatisticsGen — Analisis Statistik Data

Menghitung statistik deskriptif dari seluruh fitur: distribusi nilai, 
jumlah missing values, dan frekuensi tiap kategori. Output digunakan 
oleh SchemaGen dan ExampleValidator.

In [4]:
statistics_gen = StatisticsGen(
    examples=example_gen.outputs["examples"]
)
interactive_context.run(statistics_gen)
interactive_context.show(statistics_gen.outputs["statistics"])

## 3. SchemaGen — Inferensi Skema Data

Membuat skema otomatis berisi tipe data tiap fitur, domain nilai valid 
untuk fitur kategorikal, dan kardinalitas fitur. Skema ini menjadi 
kontrak data yang menjamin konsistensi antara training dan serving.

In [5]:
schema_gen = SchemaGen(
    statistics=statistics_gen.outputs["statistics"]
)
interactive_context.run(schema_gen)
interactive_context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'bruises',STRING,required,,'bruises'
'cap-color',STRING,required,,'cap-color'
'cap-shape',STRING,required,,'cap-shape'
'cap-surface',STRING,required,,'cap-surface'
'class',STRING,required,,'class'
'gill-attachment',STRING,required,,'gill-attachment'
'gill-color',STRING,required,,'gill-color'
'gill-size',STRING,required,,'gill-size'
'gill-spacing',STRING,required,,'gill-spacing'


,Values
Domain,
'bruises',"'f', 't'"
'cap-color',"'b', 'c', 'e', 'g', 'n', 'p', 'r', 'u', 'w', 'y'"
'cap-shape',"'b', 'c', 'f', 'k', 's', 'x'"
'cap-surface',"'f', 'g', 's', 'y'"
'class',"'e', 'p'"
'gill-attachment',"'a', 'f'"
'gill-color',"'b', 'e', 'g', 'h', 'k', 'n', 'o', 'p', 'r', 'u', 'w', 'y'"
'gill-size',"'b', 'n'"
'gill-spacing',"'c', 'w'"


## 4. ExampleValidator — Validasi Kualitas Data

Mendeteksi anomali dengan membandingkan statistik aktual terhadap skema. 
Anomali yang dideteksi: nilai di luar domain, missing values, dan 
distribusi skew antara split train dan eval.

In [6]:
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"],
    schema=schema_gen.outputs["schema"],
)
interactive_context.run(example_validator)
interactive_context.show(example_validator.outputs["anomalies"])

## 5. Transform — Feature Engineering

Preprocessing menggunakan `preprocessing_fn` di `mushroom_transform.py`:
- **22 fitur kategorikal** dikonversi ke integer via `tft.compute_and_apply_vocabulary()`
- **Label** `class` → biner: `p`=1 (beracun), `e`=0 (aman)
- Semua fitur hasil transformasi diberi sufiks `_xf`
- Transform graph memastikan preprocessing identik antara training dan serving

In [7]:
transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE),
)
interactive_context.run(transform)

Instructions for updating:
Use ref() instead.


Instructions for updating:
Use ref() instead.


INFO:tensorflow:Assets written to: yogidharma-pipeline\Transform\transform_graph\26\.temp_path\tftransform_tmp\bc2cd309b8904500a979f425994bea56\assets


INFO:tensorflow:Assets written to: yogidharma-pipeline\Transform\transform_graph\26\.temp_path\tftransform_tmp\bc2cd309b8904500a979f425994bea56\assets


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


ExecutionResult(
    component_id: Transform
    execution_id: 26
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

## 6. Tuner — Hyperparameter Tuning Otomatis

Mencari kombinasi hyperparameter optimal menggunakan **KerasTuner Hyperband**.

| Hyperparameter | Search Space |
|---|---|
| `learning_rate` | {1e-4, 5e-4, 1e-3} |
| `dense_1_units` | 32 – 256 (step 32) |
| `dense_2_units` | 16 – 128 (step 16) |
| `dropout_rate` | 0.1 – 0.5 (step 0.1) |

Objektif: **maximize val_auc**. Hasil terbaik diteruskan ke Trainer.

In [9]:
module_file = os.path.abspath( os.path.join("modules", "mushroom_tuner.py") )

tuner = Tuner(
    module_file=module_file, 
    examples=transform.outputs['transformed_examples'], 
    transform_graph=transform.outputs['transform_graph'], 
    train_args=trainer_pb2.TrainArgs(num_steps=1000), 
    eval_args=trainer_pb2.EvalArgs(num_steps=500), 
)

interactive_context.run(tuner)

Trial 5 Complete [00h 00m 40s]
val_auc: 0.9998805522918701

Best val_auc So Far: 1.0
Total elapsed time: 00h 03m 45s
Results summary
Results in yogidharma-pipeline\.temp\28\mushroom_tuning
Showing 10 best trials
Objective(name="val_auc", direction="max")

Trial 1 summary
Hyperparameters:
learning_rate: 0.001
dense_1_units: 224
dense_2_units: 128
dropout_rate: 0.5
Score: 1.0

Trial 0 summary
Hyperparameters:
learning_rate: 0.0005
dense_1_units: 32
dense_2_units: 128
dropout_rate: 0.4
Score: 0.999921441078186

Trial 4 summary
Hyperparameters:
learning_rate: 0.0001
dense_1_units: 224
dense_2_units: 32
dropout_rate: 0.2
Score: 0.9998805522918701

Trial 2 summary
Hyperparameters:
learning_rate: 0.0001
dense_1_units: 96
dense_2_units: 112
dropout_rate: 0.30000000000000004
Score: 0.998589277267456

Trial 3 summary
Hyperparameters:
learning_rate: 0.0001
dense_1_units: 160
dense_2_units: 32
dropout_rate: 0.5
Score: 0.9957499504089355


ExecutionResult(
    component_id: Tuner
    execution_id: 28
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={})
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}))

## 7. Trainer — Pelatihan Model

Melatih model MLP menggunakan hyperparameter terbaik dari Tuner.

**Arsitektur:** Input (22 fitur) → Concatenate → Dense(HP) → Dropout(HP) → Dense(HP) → Sigmoid  
**Optimizer:** Adam | **Loss:** Binary Crossentropy  
**Callbacks:** EarlyStopping (monitor val_auc, patience=3)  
**Output:** SavedModel dengan serving signature untuk TF Serving

In [17]:
trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE_FILE),
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(num_steps=1000),
    eval_args=trainer_pb2.EvalArgs(num_steps=500),
)
interactive_context.run(trainer)


Menggunakan hyperparameter dari tuner: {'learning_rate': 0.001, 'dense_1_units': 64, 'dense_2_units': 32, 'dropout_rate': 0.2}

Epoch 1/10
 994/1000 [============================>.] - ETA: 0s - loss: 0.0991 - accuracy: 0.9647 - auc: 0.9944 - precision: 0.9626 - recall: 0.9637WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 500 batches). You may need to use the repeat() function when building your dataset.


1000/1000 [==============================] - 17s 14ms/step - loss: 0.0987 - accuracy: 0.9648 - auc: 0.9944 - precision: 0.9628 - recall: 0.9638 - val_loss: 0.0052 - val_accuracy: 0.9994 - val_auc: 1.0000 - val_precision: 1.0000 - val_recall: 0.9987
Epoch 2/10
 995/1000 [============================>.] - ETA: 0s - loss: 0.0078 - accuracy: 0.9984 - auc: 0.9999 - precision: 0.9986 - recall: 0.9982WARNING:tensorflow:Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: loss,accuracy,auc,precision,recall


1000/1000 [==============================] - 8s 8ms/step - loss: 0.0077 - accuracy: 0.9984 - auc: 0.9999 - precision: 0.9986 - recall: 0.9982
Epoch 3/10
  47/1000 [>.............................] - ETA: 6s - loss: 0.0041 - accuracy: 0.9993 - auc: 1.0000 - precision: 1.0000 - recall: 0.9986WARNING:tensorflow:Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches (in this case, 10000 batches). You may need to use the repeat() function when building your dataset.


1000/1000 [==============================] - 1s 533us/step - loss: 0.0042 - accuracy: 0.9994 - auc: 1.0000 - precision: 1.0000 - recall: 0.9987
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


...

INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: yogidharma-pipeline\Trainer\model\31\Format-Serving\assets


INFO:tensorflow:Assets written to: yogidharma-pipeline\Trainer\model\31\Format-Serving\assets



Model berhasil disimpan di: yogidharma-pipeline\Trainer\model\31\Format-Serving



ExecutionResult(
    component_id: Trainer
    execution_id: 31
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

## 8. Evaluator — Evaluasi & Validasi Model

Mengevaluasi model menggunakan TFMA. Model mendapat status **BLESSED** jika:
- **BinaryAccuracy ≥ 0.80**
- **AUC ≥ 0.80**

Model yang BLESSED akan di-deploy oleh Pusher. Model yang tidak memenuhi 
threshold mendapat status NOT BLESSED dan tidak akan di-deploy.

In [18]:
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('Latest_blessed_model_resolver')

interactive_context.run(model_resolver)

ExecutionResult(
    component_id: Latest_blessed_model_resolver
    execution_id: 32
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

## 9. Pusher — Deployment Model

Men-deploy model BLESSED ke `serving_model_dir` dalam format **TensorFlow SavedModel**.
Model hanya di-push jika lolos evaluasi. Setiap versi disimpan dalam subfolder 
bertimestamp untuk mendukung model versioning.

Model yang di-push siap digunakan via **TensorFlow Serving + Docker** 
pada endpoint `http://localhost:8501/v1/models/mushroom_model:predict`.

In [19]:
eval_config = tfma.EvalConfig(
    model_specs=[
        tfma.ModelSpec(label_key='class_xf')
    ],
    slicing_specs=[
        tfma.SlicingSpec()
    ],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name='ExampleCount'),
                tfma.MetricConfig(
                    class_name='AUC',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.8}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -1e-10}
                        )
                    )
                ),
                tfma.MetricConfig(class_name='Precision'),
                tfma.MetricConfig(class_name='Recall'),
                tfma.MetricConfig(
                    class_name='BinaryAccuracy',
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={'value': 0.8}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={'value': -1e-10}
                        )
                    )
                ),
            ]
        )
    ]
)

evaluator = Evaluator(
    examples=transform.outputs['transformed_examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)
interactive_context.run(evaluator)


Failed at key ['cap-shape_xf'] in {'cap-shape_xf': array([0], dtype=int64), 'cap-surface_xf': array([1], dtype=int64), 'cap-color_xf': array([3], dtype=int64), 'bruises_xf': array([1], dtype=int64), 'odor_xf': array([4], dtype=int64), 'gill-attachment_xf': array([0], dtype=int64), 'gill-spacing_xf': array([0], dtype=int64), 'gill-size_xf': array([0], dtype=int64), 'gill-color_xf': array([7], dtype=int64), 'stalk-shape_xf': array([1], dtype=int64), 'stalk-root_xf': array([3], dtype=int64), 'stalk-surface-above-ring_xf': array([0], dtype=int64), 'stalk-surface-below-ring_xf': array([0], dtype=int64), 'stalk-color-above-ring_xf': array([0], dtype=int64), 'stalk-color-below-ring_xf': array([0], dtype=int64), 'veil-type_xf': array([0], dtype=int64), 'veil-color_xf': array([0], dtype=int64), 'ring-number_xf': array([0], dtype=int64), 'ring-type_xf': array([0], dtype=int64), 'spore-print-color_xf': array([1], dtype=int64), 'population_xf': array([3], dtype=int64), 'habitat_xf': array([1], dt


Failed at key ['bruises_xf'] in {'cap-shape_xf': array([0], dtype=int64), 'cap-surface_xf': array([1], dtype=int64), 'cap-color_xf': array([3], dtype=int64), 'bruises_xf': array([1], dtype=int64), 'odor_xf': array([4], dtype=int64), 'gill-attachment_xf': array([0], dtype=int64), 'gill-spacing_xf': array([0], dtype=int64), 'gill-size_xf': array([0], dtype=int64), 'gill-color_xf': array([7], dtype=int64), 'stalk-shape_xf': array([1], dtype=int64), 'stalk-root_xf': array([3], dtype=int64), 'stalk-surface-above-ring_xf': array([0], dtype=int64), 'stalk-surface-below-ring_xf': array([0], dtype=int64), 'stalk-color-above-ring_xf': array([0], dtype=int64), 'stalk-color-below-ring_xf': array([0], dtype=int64), 'veil-type_xf': array([0], dtype=int64), 'veil-color_xf': array([0], dtype=int64), 'ring-number_xf': array([0], dtype=int64), 'ring-type_xf': array([0], dtype=int64), 'spore-print-color_xf': array([1], dtype=int64), 'population_xf': array([3], dtype=int64), 'habitat_xf': array([1], dtyp


Failed at key ['gill-spacing_xf'] in {'cap-shape_xf': array([0], dtype=int64), 'cap-surface_xf': array([1], dtype=int64), 'cap-color_xf': array([3], dtype=int64), 'bruises_xf': array([1], dtype=int64), 'odor_xf': array([4], dtype=int64), 'gill-attachment_xf': array([0], dtype=int64), 'gill-spacing_xf': array([0], dtype=int64), 'gill-size_xf': array([0], dtype=int64), 'gill-color_xf': array([7], dtype=int64), 'stalk-shape_xf': array([1], dtype=int64), 'stalk-root_xf': array([3], dtype=int64), 'stalk-surface-above-ring_xf': array([0], dtype=int64), 'stalk-surface-below-ring_xf': array([0], dtype=int64), 'stalk-color-above-ring_xf': array([0], dtype=int64), 'stalk-color-below-ring_xf': array([0], dtype=int64), 'veil-type_xf': array([0], dtype=int64), 'veil-color_xf': array([0], dtype=int64), 'ring-number_xf': array([0], dtype=int64), 'ring-type_xf': array([0], dtype=int64), 'spore-print-color_xf': array([1], dtype=int64), 'population_xf': array([3], dtype=int64), 'habitat_xf': array([1],


Failed at key ['stalk-shape_xf'] in {'cap-shape_xf': array([0], dtype=int64), 'cap-surface_xf': array([1], dtype=int64), 'cap-color_xf': array([3], dtype=int64), 'bruises_xf': array([1], dtype=int64), 'odor_xf': array([4], dtype=int64), 'gill-attachment_xf': array([0], dtype=int64), 'gill-spacing_xf': array([0], dtype=int64), 'gill-size_xf': array([0], dtype=int64), 'gill-color_xf': array([7], dtype=int64), 'stalk-shape_xf': array([1], dtype=int64), 'stalk-root_xf': array([3], dtype=int64), 'stalk-surface-above-ring_xf': array([0], dtype=int64), 'stalk-surface-below-ring_xf': array([0], dtype=int64), 'stalk-color-above-ring_xf': array([0], dtype=int64), 'stalk-color-below-ring_xf': array([0], dtype=int64), 'veil-type_xf': array([0], dtype=int64), 'veil-color_xf': array([0], dtype=int64), 'ring-number_xf': array([0], dtype=int64), 'ring-type_xf': array([0], dtype=int64), 'spore-print-color_xf': array([1], dtype=int64), 'population_xf': array([3], dtype=int64), 'habitat_xf': array([1], 


Failed at key ['stalk-surface-below-ring_xf'] in {'cap-shape_xf': array([0], dtype=int64), 'cap-surface_xf': array([1], dtype=int64), 'cap-color_xf': array([3], dtype=int64), 'bruises_xf': array([1], dtype=int64), 'odor_xf': array([4], dtype=int64), 'gill-attachment_xf': array([0], dtype=int64), 'gill-spacing_xf': array([0], dtype=int64), 'gill-size_xf': array([0], dtype=int64), 'gill-color_xf': array([7], dtype=int64), 'stalk-shape_xf': array([1], dtype=int64), 'stalk-root_xf': array([3], dtype=int64), 'stalk-surface-above-ring_xf': array([0], dtype=int64), 'stalk-surface-below-ring_xf': array([0], dtype=int64), 'stalk-color-above-ring_xf': array([0], dtype=int64), 'stalk-color-below-ring_xf': array([0], dtype=int64), 'veil-type_xf': array([0], dtype=int64), 'veil-color_xf': array([0], dtype=int64), 'ring-number_xf': array([0], dtype=int64), 'ring-type_xf': array([0], dtype=int64), 'spore-print-color_xf': array([1], dtype=int64), 'population_xf': array([3], dtype=int64), 'habitat_xf'

...

[[1]
 [0]
 [1]
 [1]
 [1]
 [0]
 [0]
 [1]
 [1]
 [1]
 [0]
 [0]
 [1]
 [0]
 [1]
 [0]
 [1]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]
 [0]
 [1]
 [0]
 [1]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [1]
 [0]
 [0]
 [0]
 [4]
 [1]], shape=(39, 1), dtype=int64) is not compatible with TensorSpec(shape=(None, 1), dtype=tf.float32, name='ring-type_xf') 

Failed at key ['ring-type_xf'] in {'cap-shape_xf': array([2, 3, 2, 2, 1, 3, 2, 2, 2, 2, 3, 1, 0, 3, 2, 1, 0, 3, 3, 2, 2, 2,
       2, 2, 2, 1, 2, 2, 3, 1, 2, 2, 2, 2, 2, 0, 2, 1, 2], dtype=int64), 'cap-surface_xf': array([0, 2, 0, 0, 0, 2, 1, 1, 1, 0, 1, 1, 0, 2, 1, 1, 1, 2, 2, 1, 1, 0,
       1, 2, 1, 1, 0, 1, 2, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0], dtype=int64), 'cap-color_xf': array([2, 4, 2, 0, 0, 4, 0, 2, 0, 0, 0, 0, 2, 4, 2, 0, 0, 4, 1, 0, 0, 0,
       0, 4, 2, 0, 2, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 7, 0], dtype=int64), 'bruises_xf': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64), 

[[0]
 [3]
 [0]
 [0]
 [0]
 [3]
 [0]
 [0]
 [0]
 [0]
 [5]
 [0]
 [0]
 [3]
 [0]
 [0]
 [0]
 [3]
 [3]
 [0]
 [0]
 [0]
 [0]
 [2]
 [0]
 [5]
 [0]
 [0]
 [3]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [5]
 [0]
 [5]
 [0]], shape=(39, 1), dtype=int64) is not compatible with TensorSpec(shape=(None, 1), dtype=tf.float32, name='population_xf') 

Failed at key ['population_xf'] in {'cap-shape_xf': array([2, 3, 2, 2, 1, 3, 2, 2, 2, 2, 3, 1, 0, 3, 2, 1, 0, 3, 3, 2, 2, 2,
       2, 2, 2, 1, 2, 2, 3, 1, 2, 2, 2, 2, 2, 0, 2, 1, 2], dtype=int64), 'cap-surface_xf': array([0, 2, 0, 0, 0, 2, 1, 1, 1, 0, 1, 1, 0, 2, 1, 1, 1, 2, 2, 1, 1, 0,
       1, 2, 1, 1, 0, 1, 2, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0], dtype=int64), 'cap-color_xf': array([2, 4, 2, 0, 0, 4, 0, 2, 0, 0, 0, 0, 2, 4, 2, 0, 0, 4, 1, 0, 0, 0,
       0, 4, 2, 0, 2, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 7, 0], dtype=int64), 'bruises_xf': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=int64)

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


ExecutionResult(
    component_id: Evaluator
    execution_id: 33
    outputs:
        evaluation: OutputChannel(artifact_type=ModelEvaluation, producer_component_id=Evaluator, output_key=evaluation, additional_properties={}, additional_custom_properties={})
        blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=Evaluator, output_key=blessing, additional_properties={}, additional_custom_properties={}))

In [20]:
eval_result = evaluator.outputs['evaluation'].get()[0].uri
tfma_result = tfma.load_eval_result(eval_result)
tfma.view.render_slicing_metrics(tfma_result)

SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

In [21]:
# Cek status blessing model
blessing = evaluator.outputs['blessing'].get()[0]
blessed_file     = os.path.join(blessing.uri, 'BLESSED')
not_blessed_file = os.path.join(blessing.uri, 'NOT_BLESSED')

if os.path.exists(blessed_file):
    print('✅ Model Status: BLESSED — model siap untuk di-deploy!')
elif os.path.exists(not_blessed_file):
    print('❌ Model Status: NOT BLESSED — model tidak memenuhi threshold.')
else:
    print('⚠️  Status tidak diketahui, cek folder:', blessing.uri)

✅ Model Status: BLESSED — model siap untuk di-deploy!


## 10. Pusher — Deployment Model ke Serving Directory

Pusher hanya deploy model yang mendapat status **BLESSED** dari Evaluator.

In [22]:
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
interactive_context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 34
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))

In [23]:
# Verifikasi model telah di-push
pushed_versions = os.listdir(SERVING_MODEL_DIR) if os.path.exists(SERVING_MODEL_DIR) else []
if pushed_versions:
    print(f"✅ Model berhasil di-push. Versi tersedia: {sorted(pushed_versions)}")
    print(f"   Path serving: {SERVING_MODEL_DIR}/{sorted(pushed_versions)[-1]}")
else:
    print("⚠️  Belum ada model yang di-push (model mungkin NOT BLESSED)")

✅ Model berhasil di-push. Versi tersedia: ['1779535583']
   Path serving: serving_model_dir/mushroom_model/1779535583
